In [1]:
import openai
import json
import pandas as pd
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from copy import deepcopy
import threading

# 添加线程锁保护共享资源
# 配置OpenAI API[citation:1]
openai.api_key = "REDACTED_API_KEY"  # 建议使用环境变量管理
# 或者使用较新版本的客户端[citation:10]
from openai import OpenAI
chatclient = OpenAI(api_key="REDACTED_API_KEY",
                base_url="https://api.deepseek.com/v1")
MODEL = "deepseek-chat"
NUM_FACTORS = 5 


ar_functions = {
    "abs(x)": "Absolute value of x",
    "add(x, y, filter = false), x + y": "Add all inputs (at least 2 inputs required). If filter = true, filter all input NaN to 0 before adding",
    "densify(x)": "Converts a grouping field of many buckets into lesser number of only available buckets so as to make working with grouping fields computationally efficient",
    "divide(x, y), x / y": "x / y",
    "inverse(x)": "1 / x",
    "log(x)": "Natural logarithm. For example: Log(high/low) uses natural logarithm of high/low ratio as stock weights.",
    "max(x, y, ..)": "Maximum value of all inputs. At least 2 inputs are required",
    "min(x, y, ..)": "Minimum value of all inputs. At least 2 inputs are required",
    "multiply(x, y, ..., filter=false), x * y": "Multiply all inputs. At least 2 inputs are required. Filter sets the NaN values to 1",
    "reverse(x)":"-x"
}

cs_functions = {
    "normalize(x, useStd = false, limit = 0.0)": "Calculates the mean value of all valid alpha values for a certain date, then subtracts that mean from each element",
    "quantile(x, driver = gaussian, sigma = 1.0)": "Rank the raw vector, shift the ranked Alpha vector, apply distribution (gaussian, cauchy, uniform). If driver is uniform, it simply subtract each Alpha value with the mean of all Alpha values in the Alpha vector",
    "rank(x, rate=2)": "Ranks the input among all the instruments and returns an equally distributed number between 0.0 and 1.0. For precise sort, use the rate as 0",
    "scale(x, scale=1, longscale=1, shortscale=1)": "Scale's input to booksize. We can also scale the long positions and short positions to separate scales by mentioning additional parameters to the operator",
    "truncate(x,maxPercent =0.01)": "Operator truncates all values of x to maxPercent. Here, maxPercent is in decimal notation",
    "vector_neut(x, y)": "For given vectors x and y, it finds a new vector x* (output) such that x* is orthogonal to y",
    "winsorize(x, std=4)": "Winsorizes x to make sure that all values in x are between the lower and upper limits, which are specified as multiple of std.",
    "zscore(x)": "Z-score is a numerical measurement that describes a value's relationship to the mean of a group of values. Z-score is measured in terms of standard deviations from the mean"
}

ts_functions = {
    "days_from_last_change(x)": "Amount of days since last change of x",
    "hump(x, hump = 0.01)": "Limits amount and magnitude of changes in input (thus reducing turnover)",
    "inst_tvr(x, d)": "Total trading value / Total holding value in the past d days",
    "kth_element(x, d, k)": "Returns k-th value of input by looking through lookback days. This operator can be used to backfill missing data if k=1",
    "last_diff_value(x, d)": "Returns last x value not equal to current x value from last d days",
    "ts_arg_max(x, d)": "Returns the relative index of the max value in the time series for the past d days. If the current day has the max value for the past d days, it returns 0. If previous day has the max value for the past d days, it returns 1",
    "ts_arg_min(x, d)": "Returns the relative index of the min value in the time series for the past d days; if the current day has the min value for the past d days, it returns 0; if previous day has the min value for the past d days, it returns 1.",
    "ts_backfill(x,lookback = d, k=1, ignore='NAN')": "Backfill is the process of replacing the NAN or 0 values by a meaningful value (i.e., a first non-NaN value)",
    "ts_corr(x, y, d)": "Returns correlation of x and y for the past d days",
    "ts_count_nans(x, d)": "Returns the number of NaN values in x for the past d days",
    "ts_covariance(y, x, d)": "Returns covariance of y and x for the past d days",
    "ts_decay_exp_window(x, d, factor = f)": "Returns exponential decay of x with smoothing factor for the past d days",
    "ts_decay_linear(x, d, dense = false)": "Returns the linear decay on x for the past d days. Dense parameter=false means operator works in sparse mode and we treat NaN as 0. In dense mode we do not.",
    "ts_delay(x, d)": "Returns x value d days ago",
    "ts_delta(x, d)": "Returns x - ts_delay(x, d)",
    "ts_delta_limit(x, y, limit_volume=0.1)": "Limit the change in the Alpha position x between dates to a specified fraction of y. The 'limit_volume' can be in the range of 0 to 1. Also, please be aware of the scaling for x and y. Besides setting y as adv20 or volume related data, you can also set y as a constant.",
    "ts_mean(x, d)": "Returns average value of x for the past d days.",
    "ts_median(x, d)": "Returns median value of x for the past d days",
    "ts_product(x, d)": "Returns product of x for the past d days",
    "ts_quantile(x,d, driver='gaussian')": "It calculates ts_rank and apply to its value an inverse cumulative density function from driver distribution. Possible values of driver (optional ) are 'gaussian', 'uniform', 'cauchy' distribution where 'gaussian' is the default.",
    "ts_regression(y, x, d, lag = 0, rettype = 0)": "Returns various parameters related to regression function",
    "ts_scale(x, d, constant = 0)": "Returns (x - ts_min(x, d)) / (ts_max(x, d) - ts_min(x, d)) + constant. This operator is similar to scale down operator but acts in time series space",
    "ts_std_dev(x, d)": "Returns standard deviation of x for the past d days",
    "ts_step(1)": "Returns days' counter",
    "ts_sum(x, d)": "Sum values of x for the past d days.",
    "ts_target_tvr_decay(x, lambda_min=0, lambda_max=1, target_tvr=0.1)": "True 'ts_decay' to have a turnover equal to a certain target, with optimization weight range between lambda_min, lambda_max",
    "ts_zscore(x, d)": "Z-score is a numerical measurement that describes a value's relationship to the mean of a group of values. Z-score is measured in terms of standard deviations from the mean: (x - tsmean(x,d)) / tsstddev(x,d). This operator may help reduce outliers and drawdown."
}

In [2]:
from brain_batch_alpha_change_mode import BrainBatchAlpha
from copy import deepcopy
client = BrainBatchAlpha("brain_credentials.txt")


alpha_records = []

def simulate_with_auto_reauth(client, alpha, credentials_file='brain_credentials.txt', max_retry=1):
    for attempt in range(max_retry + 1):
        result = client._simulate_single_alpha(alpha)
        # 检查是否因认证失败导致的 401/403
        if result is None and hasattr(client, 'last_status_code'):
            if client.last_status_code in [401, 403]:
                print("⚠️ 检测到认证失效，正在重新登录并重试...")
                client.reauthenticate(credentials_file)
                continue
        return result
    print("❌ 多次尝试后仍失败")
    return None


def check_all_passes(result):
    """
    检查result的checks里面的所有项的result是否都是PASS或者PENDING

    参数:
        result: API返回的结果字典

    返回:
        bool: 如果所有检查都是PASS或PENDING则返回True，否则返回False
        dict: 包含详细信息的字典
    """
    if not result:
        return False, {"error": "result为None或空"}

    if 'metrics' not in result:
        return False, {"error": "result中没有metrics字段"}

    if 'checks' not in result['metrics']:
        return False, {"error": "result['metrics']中没有checks字段"}

    checks = result['metrics']['checks']
    failed_checks = []
    passed_checks = []
    pending_checks = []

    for i, check in enumerate(checks):
        if 'result' not in check:
            failed_checks.append(f"检查{i}: 没有result字段")
            continue

        check_result = check['result']
        check_name = check.get('name', f'检查{i}')

        if check_result == 'PASS':
            passed_checks.append(check_name)
        elif check_result == 'PENDING':
            pending_checks.append(check_name)
        else:
            failed_checks.append(f"{check_name}: {check_result}")

    all_pass = len(failed_checks) == 0

    summary = {
        "all_pass": all_pass,
        "total_checks": len(checks),
        "passed": len(passed_checks),
        "pending": len(pending_checks),
        "failed": len(failed_checks),
        "passed_checks": passed_checks,
        "pending_checks": pending_checks,
        "failed_checks": failed_checks
    }

    return all_pass, summary

def evaluate_fitness(client,expr_str):
    """评估适应度"""
    expr = expr_str

    # 显示当前线程信息
    import threading
    print(f"🔗 线程 {threading.current_thread().name} 正在评估: {expr[:30]}{'...' if len(expr) > 30 else ''}")

    # 构造API请求参数
    alpha_config = {
        'type': 'REGULAR',
        'settings': {
            'instrumentType': 'EQUITY',
            'region': 'USA',
            'universe': 'TOP1000',
            'delay': 1,
            'decay': 3,
            'neutralization': 'SUBINDUSTRY',
            'truncation': 0.08,
            'pasteurization': 'ON',
            'unitHandling': 'VERIFY',
            'nanHandling': 'ON',
            'language': 'FASTEXPR',
            'visualization': False,
        },
        'regular': expr
    }

    # 调用API获取评估结果
    result = simulate_with_auto_reauth(client, alpha_config)

    if result is None:
        return float('-inf')

    if result['metrics']['longCount'] == 0 or result['metrics']['shortCount'] == 0:
        return float('-inf')

    # 使用新的检查函数
    all_pass, check_details = check_all_passes(result)

    # 计算失败检查数量
    failed_count = sum(1 for check in result['metrics']['checks'] if check.get('result') not in ['PASS', 'PENDING'])

    # 输出每个因子的检查结果统计
    passed_count = sum(1 for check in result['metrics']['checks'] if check.get('result') == 'PASS')
    pending_count = sum(1 for check in result['metrics']['checks'] if check.get('result') == 'PENDING')
    total_count = len(result['metrics']['checks'])

    print(f"因子: {expr[:50]}{'...' if len(expr) > 50 else ''}")
    print(f"  检查结果: 通过={passed_count}, 待定={pending_count}, 失败={failed_count}, 总计={total_count}")
    print(f"  适应度指标: Sharpe={result['metrics']['sharpe']:.4f}, Fitness={result['metrics']['fitness']:.4f}")

    # 存储失败检查数量<=1的因子信息
    if failed_count <= 1:
        print(f"  ✓ 已保存到记录 (失败检查≤1)")
        alpha_info = {
            'expression': expr,
            'alpha_id': result.get('alpha_id', 'N/A'),  # 安全访问，避免KeyError
            'sharpe': result['metrics']['sharpe'],
            'fitness': result['metrics']['fitness'],
            'margin': result['metrics']['margin'],
            'turnover': result['metrics']['turnover'],
            'longCount': result['metrics']['longCount'],
            'shortCount': result['metrics']['shortCount'],
            'concentration': result['metrics']['checks'][4]['result'] if len(
                result['metrics']['checks']) > 4 else 'N/A',
            'sub_uni_sharpe': result['metrics']['checks'][5].get('value', 'ERROR') if len(
                result['metrics']['checks']) > 5 else 'N/A',
            'sub_uni_limit': result['metrics']['checks'][5].get('limit', 'ERROR') if len(
                result['metrics']['checks']) > 5 else 'N/A',
            'all_pass': all_pass,
            'total_checks': total_count,
            'passed_checks': passed_count,
            'pending_checks': pending_count,
            'failed_checks': failed_count,
        }
        global alpha_records
        alpha_records.append(alpha_info)
        # self._write_to_csv(alpha_info)  # 写入单条记录到CSV
    else:
        print(f"  ✗ 未保存 (失败检查={failed_count}>1)")

    print()  # 空行分隔

    # 计算适应度分数
    
    sharpe = result['metrics']['sharpe']
    fitness = result['metrics']['fitness']
    turnover = result['metrics']['turnover']

    if (sharpe is None) or (fitness is None) or (turnover is None):
        return float('-inf')

    # # 综合评分
    # score = (abs(sharpe) * 0.4 +
    #             abs(fitness) * 0.4 +
    #             (1 / (1 + turnover)) * 0.2)

    return fitness,sharpe,turnover  # 返回带惩罚的适应度

✅ 认证成功!


In [3]:
def build_system_prompt(
    data_columns: list,
    ar_functions: dict,
    ts_functions: dict,
    cs_functions: dict,
    target_instruments: str,
    frequency: str = '1day',
):
    systemprompt = \
    f"""
    你是一个专业的量化因子挖掘专家，擅长发现金融市场中的有效alpha因子。请遵循以下原则：
    可用算术算子函数库：{ar_functions}
    可用时序算子函数库：{ts_functions}
    可用截面算子函数库：{cs_functions}
    基础数据字段：{data_columns}
    目标标的：{target_instruments}
    频率:{frequency}

    核心能力：
    - 基于给定的基础数据和特征库生成创新的因子表达式
    - 理解因子的潜在经济逻辑,统计特性和归类但不要求完全的可解释性
    - 预期对收益率的相关性为正（即对负向因子进行正向化处理）
    - 根据因子表现进行优化和调整因子表达式
    - 考虑风险控制和过拟合问题

    输出要求：
    - 请用结构化格式输出，确保可解析性。
    - 因子表达式必须可执行且语法正确
    - 每个因子需说明其经济逻辑
    - 评估因子的预期特性（方向性、频率等）
    - 考虑实际交易中的可行性
    """
    return {
        "role":"system", 
        "content": systemprompt
            }

In [4]:
def build_user_prompt(
    num_factors: int,
):
    """
    Build the initial system prompt for the LLM interaction.
    Returns:
        str: The system prompt string.
    """


    userprompt = \
    """
    现在，基于以下信息，生成{num_factors}个创新因子：

    【生成要求】
    - 因子表达式必须使用可用函数库和基础数据字段
    - 每个因子包含完整的经济逻辑说明
    - 评估预期表现特性

    请按以下JSON格式输出，你的返回结果将会直接被计算机进行读取，请确保语法正确，
    注意校对与要求的一致性：
    [{{
        "factor_id": "F001",
        "name": "因子名称",
        "expression": "可执行的表达式",
        "economic_logic": "经济逻辑说明",
        "complexity[one_to_ten]": 4,
        "novelty[one_to_ten]": 7
    }}]
    """.format(num_factors=num_factors)

    return {"role": "user", "content": userprompt}

In [5]:
def build_optimize_prompt(
    previous_factors: list,
):
    optimize_prompt =\
    """
    现在,你将会收到之前生成的因子列表: {previous_factors}
    这个新的JSON中，不仅包含了每个因子原先的信息字典，还含有从历史训练集中提取
    的因子表现三项指标：
    'evaluated_fitness'，适应度，范围从-1到1，1表示表现最佳
    'evaluated_sharpe'，夏普比率，数值越高表示风险调整后的收益越好，记住：如果出现小于-1，应当正向化处理，如使用reverse函数
    'evaluated_turnover'，换手率，数值越低表示交易成本越低,同等情况下优换手率低的因子更好
    
    请基于'evaluated_fitness'，'evaluated_sharpe'，'evaluated_turnover'，complexity,novelty 三项对因子进行优化和调整，提升其预期表现.


    【优化要求】
    1. 多样化：经过修改后,因子不应当过度频繁出现特定的片段或模式
    2. 提升表现：'evaluated_fitness'，'evaluated_sharpe'，'evaluated_turnover'是重要的参考指标，尝试提升低分因子的预期表现，可以从高分因子中获取灵感，
       但为了避免对【优化要求】1的违反，你可以考虑从你的自身知识中提取新的思路
    3. 因子池的整体质量：确保优化后的因子池在复杂度和新颖性上有良好的分布，但不以牺牲【优化要求】1和2为代价
    4. 经过修改，因子表达式必须保持可执行且语法正确，但经济逻辑说明可能会发生一定变化，请确保其与新的表达式一致，
       并且仍然合理，但不违背【优化要求】1-3
    请按以下JSON格式输出，注意你的输出不应包含'evaluated_fitness'，'evaluated_sharpe'，'evaluated_turnover'，你的返回结果将会直接被计算机进行读取，请确保语法正确,
    注意校对与要求的一致性：
    [{{
        "factor_id": "F001",
        "name": "因子名称",
        "expression": "可执行的表达式",
        "economic_logic": "经济逻辑说明",
        "complexity[one_to_ten]": 4,
        "novelty[one_to_ten]": 7
    }}]

    """.format(previous_factors = previous_factors)
    
    return {
        "role":"user", 
        "content": optimize_prompt
        }

In [6]:
# response = client.chat.completions.create(
#             model=MODEL,
#             messages=[systemprompt, user_prompt],
#             temperature=0.7,
#             # response_format={"type": "json_object"}  # 强制JSON输出
#         )

# response_content = response.choices[0].message.content.lstrip("```json").rstrip("```")

In [7]:
lock = threading.Lock()
def use_wq_to_estimate(factor_json):
    
    def evaluate_single_factor(client, factor_dict, index):
        """评估单个因子的线程函数"""
        expr = factor_dict['expression']
        print(f"🔗 线程 {threading.current_thread().name} 开始评估因子 {factor_dict['factor_id']}")
        
        fitness, sharpe, turnover = evaluate_fitness(client, expr)
        
        # 创建结果字典
        result = deepcopy(factor_dict)
        result['evaluated_fitness'] = fitness
        result['evaluated_sharpe'] = sharpe
        result['evaluated_turnover'] = turnover
        
        return index, result

    # 多线程评估
    next_input = deepcopy(factor_json)
    results = [None] * len(next_input)  # 预分配结果列表

    with ThreadPoolExecutor(max_workers=2) as executor:
        # 提交所有任务
        future_to_index = {
            executor.submit(evaluate_single_factor, client, next_input[i], i): i 
            for i in range(len(next_input))
        }
        
        # 收集结果
        for future in as_completed(future_to_index):
            try:
                index, result = future.result()
                results[index] = result
                print(f"✅ 因子 {result['factor_id']} 评估完成")
            except Exception as e:
                index = future_to_index[future]
                print(f"❌ 因子索引 {index} 评估失败: {str(e)}")
                # 保留原始数据，添加错误标记
                results[index] = deepcopy(next_input[index])
                results[index]['evaluated_fitness'] = float('-inf')
                results[index]['evaluated_sharpe'] = float('-inf')
                results[index]['evaluated_turnover'] = float('inf')

    # 更新 next_input
    next_input = results
    print(f"\n🎉 所有因子评估完成，共 {len(next_input)} 个")
    return next_input

In [8]:
cycle = 10
# next_input = user_prompt
systemprompt = build_system_prompt(
    data_columns=["open", "high", "low", "close", "volume"],
    target_instruments='USstocks3000',
    ts_functions=ts_functions,
    cs_functions=cs_functions,
    ar_functions=ar_functions,
    frequency='1day'
)

user_prompt = build_user_prompt(
    num_factors=NUM_FACTORS
)

for iteration in range(1,cycle):
    print(f"🔄 第 {iteration} 轮迭代")
    response = chatclient.chat.completions.create(
                model=MODEL,
                messages=[systemprompt, user_prompt],
                temperature=0.7,
                # response_format={"type": "json_object"}  # 强制JSON输出
            )
    
    response_content = response.choices[0].message.content.lstrip("```json").rstrip("```")
    factor_result = json.loads(response_content)
    print(f"📝 LLM本次返回了 {len(factor_result)} 个因子，开始评估其表现...")
    next_input = use_wq_to_estimate(factor_result)
    
    user_prompt = build_optimize_prompt(
        previous_factors=next_input
    )

print("最终结果:", next_input)

🔄 第 1 轮迭代
📝 LLM本次返回了 5 个因子，开始评估其表现...
🔗 线程 ThreadPoolExecutor-0_0 开始评估因子 F001
🔗 线程 ThreadPoolExecutor-0_0 正在评估: multiply(ts_delta(close, 5), i...
表达式: multiply(ts_delta(close, 5), inverse(ts_std_dev(close, 20)))
尝试次数: 1/3
🔗 线程 ThreadPoolExecutor-0_1 开始评估因子 F002
🔗 线程 ThreadPoolExecutor-0_1 正在评估: divide(ts_delta(close, 1), mul...
表达式: divide(ts_delta(close, 1), multiply(ts_mean(volume, 5), ts_std_dev(close, 10)))
尝试次数: 1/3
⏳ 等待模拟结果... (31.7 秒 | 进度约 53%)
⏳ 等待模拟结果... (31.8 秒 | 进度约 53%)
⏳ 等待模拟结果... (63.4 秒 | 进度约 95%)
⏳ 等待模拟结果... (63.5 秒 | 进度约 95%)
✅ 获得 Alpha ID: xA0jol2J
✅ 获得 Alpha ID: E5XV15kG
因子: multiply(ts_delta(close, 5), inverse(ts_std_dev(cl...
  检查结果: 通过=4, 待定=1, 失败=3, 总计=8
  适应度指标: Sharpe=-0.0800, Fitness=-0.0100
  ✗ 未保存 (失败检查=3>1)

🔗 线程 ThreadPoolExecutor-0_0 开始评估因子 F003
🔗 线程 ThreadPoolExecutor-0_0 正在评估: divide(ts_delta(high, 1), add(...
表达式: divide(ts_delta(high, 1), add(ts_delta(low, 1), 0.001))
尝试次数: 1/3
✅ 因子 F001 评估完成
因子: divide(ts_delta(close, 1), multiply(ts_mean(volume...
 

In [14]:
display(pd.DataFrame(next_input))

,factor_id,name,expression,economic_logic,complexity[one_to_ten],novelty[one_to_ten],evaluated_fitness,evaluated_sharpe,evaluated_turnover
0,F001,波动率调整动量效率因子,"multiply(ts_zscore(close, 10), inverse(ts_std_...",采用10日Z-score识别中期价格动量，结合12日波动率进行风险调整。优化窗口参数平衡动量...,5,7,0.18,0.41,0.4857
1,F002,量价协同突破因子,"divide(ts_delta(close, 5), multiply(ts_mean(vo...",使用5日价格变化率，结合8日成交量均值和10日波动率。优化时间窗口增强突破信号的持续性，通过...,6,6,0.03,0.16,0.4568
2,F003,价格动量稳定性因子,"multiply(ts_delta(close, 8), inverse(multiply(...",采用8日动量窗口，结合12日波动率和8日成交量均值的倒数。重新构建表达式结构，在保持动量捕捉...,6,8,-0.04,-0.17,0.3653
3,F004,波动率调整均值回归因子,"multiply(ts_zscore(close, 12), inverse(ts_std_...",使用12日Z-score识别中长期价格偏离，结合20日波动率进行风险调整。延长窗口参数增强均...,5,7,0.18,0.40,0.4241
4,F005,量价趋势确认因子,"multiply(ts_corr(ts_delta(close, 5), ts_delta(...",采用5日变化率的8日相关性，结合10日波动率的倒数。重构表达式结构，通过量价同步性验证趋势强...,7,8,-0.11,-0.30,0.3709


In [ ]:
"ts_rank(rank(reverse((multiply((ts_zscore(vwap, 10)), (ts_std_dev(vwap, 12))))))^20,20)"